In [ ]:
!sed -i 's/\r$//' prepare_data.sh
!bash prepare_data.sh

[*] Dang tai dataset (~150 MB) ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  157M  100  157M    0     0  57.0M      0  0:00:02  0:00:02 --:--:-- 62.4M
[*] Giai nen brain-tumor-mri-dataset.zip -> brain-tumor-mri-dataset ...

[+] Du lieu hop le tai /content/brain-tumor-mri-dataset
    lop             Training   Testing
    glioma              1400       400
    meningioma          1400       400
    notumor             1400       400
    pituitary           1400       400
    TONG                5600      1600
Da tao manifest trong ./partitions
client  chu dao             glioma  meningioma     notumor   pituitary
A       glioma          1080 (90.0%)    40 ( 3.3%)    40 ( 3.3%)    40 ( 3.3%)
B       pituitary         40 ( 3.3%)    40 ( 3.3%)    40 ( 3.3%)  1080 (90.0%)
Validation toan cuc: 4

In [ ]:
!python run_simulation.py --data_root ./brain-tumor-mri-dataset --tag fedprox_dp --mu 0.05 --dp --train_mode head

[Partition] Thieu manifest (partitions/client_C.json, partitions/client_D.json); tu dong chia lai du lieu.

$ /usr/bin/python3 /content/partition.py --data_root ./brain-tumor-mri-dataset --out_dir partitions --clients A:glioma B:meningioma C:notumor D:pituitary --samples_per_client 1200 --dominant_frac 0.9 --seed 42
Da tao manifest trong partitions
client  chu dao             glioma  meningioma     notumor   pituitary
A       glioma          1080 (90.0%)    40 ( 3.3%)    40 ( 3.3%)    40 ( 3.3%)
B       meningioma        40 ( 3.3%)  1080 (90.0%)    40 ( 3.3%)    40 ( 3.3%)
C       notumor           40 ( 3.3%)    40 ( 3.3%)  1080 (90.0%)    40 ( 3.3%)
D       pituitary         40 ( 3.3%)    40 ( 3.3%)    40 ( 3.3%)  1080 (90.0%)
Validation toan cuc: 400 anh (100/lop)

$ /usr/bin/python3 /content/server.py init --run_dir runs/fedprox_dp --seed 42
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047

In [ ]:
%cd /content
!python run_simulation.py --data_root ./brain-tumor-mri-dataset --tag tune_last_blocks \
  --rounds 20 --epochs 3 --batch_size 32 --lr 0.0001 --mu 0.01 \
  --train_mode last_blocks --dominant_frac 0.7 --repartition --skip_eval

/content

$ /usr/bin/python3 /content/partition.py --data_root ./brain-tumor-mri-dataset --out_dir partitions --clients A:glioma B:meningioma C:notumor D:pituitary --samples_per_client 1200 --dominant_frac 0.7 --seed 42
Da tao manifest trong partitions
client  chu dao             glioma  meningioma     notumor   pituitary
A       glioma           840 (70.0%)   120 (10.0%)   120 (10.0%)   120 (10.0%)
B       meningioma       120 (10.0%)   840 (70.0%)   120 (10.0%)   120 (10.0%)
C       notumor          120 (10.0%)   120 (10.0%)   840 (70.0%)   120 (10.0%)
D       pituitary        120 (10.0%)   120 (10.0%)   120 (10.0%)   840 (70.0%)
Validation toan cuc: 400 anh (100/lop)

$ /usr/bin/python3 /content/server.py init --run_dir runs/tune_last_blocks --seed 42
[Server] Da tao mo hinh toan cuc ban dau: runs/tune_last_blocks/global/global_round_00.pt  (pretrained=True)

================ ROUND 1/20 ================

$ /usr/bin/python3 /content/client.py --client_id A --round 1 --run_dir runs/tu

In [ ]:
!python evaluate.py --run_dir runs/tune_last_blocks --split val \
  --partition_dir partitions --all_rounds --device auto --no_plots

[Eval] val: 400 anh | thiet bi cuda

Mo hinh                                         Acc  BalAcc     F1 |  gliom  menin  notum  pitui   (recall)
---------------------------------------------------------------------------------------------------------
Server | R0 (TRUOC huan luyen)                26.25   26.25  23.35 |   21.0   45.0   35.0    4.0
Client A | R1 sau train cuc bo                85.75   85.75  85.69 |   77.0   77.0   92.0   97.0
Client B | R1 sau train cuc bo                86.25   86.25  86.29 |   80.0   80.0   93.0   92.0
Client C | R1 sau train cuc bo                82.75   82.75  82.46 |   80.0   60.0   96.0   95.0
Client D | R1 sau train cuc bo                83.25   83.25  83.25 |   70.0   77.0   89.0   97.0
Server | R1                                   86.00   86.00  85.76 |   78.0   73.0   94.0   99.0
Server | R2                                   89.50   89.50  89.39 |   79.0   83.0   97.0   99.0
Server | R3                                   90.00   90.00  89.93 |  

In [ ]:
!python evaluate.py --run_dir runs/tune_last_blocks --split test \
  --data_root ./brain-tumor-mri-dataset --server_rounds 8 --client_rounds "" \
  --device auto --no_plots

[Eval] test: 1600 anh | thiet bi cuda

Mo hinh                                         Acc  BalAcc     F1 |  gliom  menin  notum  pitui   (recall)
---------------------------------------------------------------------------------------------------------
Server | R8                                   88.62   88.62  88.34 |   72.0   85.5   99.8   97.2

[Eval] Da luu: runs/tune_last_blocks/eval/eval_test.json|csv


In [ ]:
%cd /content
!python evaluate.py --run_dir runs/tune_last_blocks --split test \
  --data_root ./brain-tumor-mri-dataset --server_rounds 18 --client_rounds "" \
  --device auto --no_plots

/content
[Eval] test: 1600 anh | thiet bi cuda

Mo hinh                                         Acc  BalAcc     F1 |  gliom  menin  notum  pitui   (recall)
---------------------------------------------------------------------------------------------------------
Server | R18                                  92.06   92.06  91.84 |   75.8   93.2  100.0   99.2

[Eval] Da luu: runs/tune_last_blocks/eval/eval_test.json|csv


In [ ]:
%cd /content
!python benchmark.py --data_root ./brain-tumor-mri-dataset \
  --tag competition_benchmark --rounds 20 --epochs 3 --batch_size 32 \
  --lr 0.0001 --mu 0.01 --train_mode last_blocks \
  --dominant_frac 0.7 --device auto

/content

$ /usr/bin/python3 /content/partition.py --data_root ./brain-tumor-mri-dataset --out_dir partitions --clients A:glioma B:meningioma C:notumor D:pituitary --samples_per_client 1200 --dominant_frac 0.7 --seed 42
Da tao manifest trong partitions
client  chu dao             glioma  meningioma     notumor   pituitary
A       glioma           840 (70.0%)   120 (10.0%)   120 (10.0%)   120 (10.0%)
B       meningioma       120 (10.0%)   840 (70.0%)   120 (10.0%)   120 (10.0%)
C       notumor          120 (10.0%)   120 (10.0%)   840 (70.0%)   120 (10.0%)
D       pituitary        120 (10.0%)   120 (10.0%)   120 (10.0%)   840 (70.0%)
Validation toan cuc: 400 anh (100/lop)

$ /usr/bin/python3 /content/run_simulation.py --data_root ./brain-tumor-mri-dataset --tag competition_benchmark_fedavg --runs_dir runs --partition_dir partitions --clients A:glioma B:meningioma C:notumor D:pituitary --samples_per_client 1200 --dominant_frac 0.7 --rounds 20 --epochs 3 --batch_size 32 --lr 0.0001 --mu 0.